# RGAN Research Project - Google Colab Demo

**Noise-Resilient Time-Series Forecasting with LSTM-based GANs**

This notebook provides a complete, runnable demonstration of the RGAN (Recurrent Generative Adversarial Network) for time-series forecasting. The project trains an LSTM generator and discriminator adversarially to produce forecasts robust to real-world noise.

## What This Notebook Does

1. **Setup**: Installs all required dependencies and configures the environment
2. **Data Source Selection**: Choose to upload your own CSV, download a real dataset, or use synthetic data
3. **Configuration**: Configure the training parameters for Colab's runtime limits
4. **Training**: Run RGAN training with your chosen data
5. **Evaluation**: Compute metrics and visualize results
6. **Optional Augmentation**: Demonstrate synthetic data generation and augmentation experiments

## Requirements

- Google Colab (free GPU)
- Internet connection (for dataset download)
- Approximately 2GB RAM, 5GB disk space

**Expected runtime**: ~10-15 minutes for the demo configuration

Let's get started!

## 1. Setup & Installation

First, we'll install all required packages and set up the project structure.

In [ ]:
# Install nbformat for notebook generation (if not already installed)
!pip install -q nbformat tqdm ipywidgets pyyaml

# Import our Colab utilities
import sys
sys.path.insert(0, '/content')

# Clone or copy the RGAN repository
import os
from pathlib import Path

# Check if we're in Colab
IN_COLAB = 'COLAB_GPU' in os.environ
if IN_COLAB:
    print("Running in Google Colab")
    # Clone the repository
    !git clone https://github.com/RandomEdge999/RGAN-Research-Project.git /content/RGAN-Research-Project 2>/dev/null || echo "Repository already exists"
    os.chdir('/content/RGAN-Research-Project')
else:
    print("Running locally")
    # Assume we're already in the project directory

In [ ]:
# Import colab utilities
from colab_utils import setup_project

# Complete project setup
setup_result = setup_project(
    use_drive=False,  # Set to True to save results to Google Drive
    download_data=False,  # We'll handle data selection in next cell
    synthetic_fallback=False
)

print(f"Project ready at: {setup_result['project_path']}")

## 2. Data Source Selection

Choose where to get your time-series data. You can upload your own CSV file, download a real-world dataset, or use synthetic data for demonstration.

In [ ]:
# ===== DATA SOURCE SELECTION =====
print("Choose your data source:")
print("1. Upload your own CSV file")
print("2. Download NASA POWER Austin dataset (requires internet)")
print("3. Use synthetic demo data (no internet required)")

choice = input("Enter choice (1, 2, or 3): ").strip()

# Variables to store user's selection
user_csv_path = None
target_column = None
time_column = None

if choice == "1":
    # Manual CSV upload
    from google.colab import files
    import pandas as pd
    
    print("\nPlease upload your CSV file when prompted...")
    uploaded = files.upload()
    
    if uploaded:
        file_name = list(uploaded.keys())[0]
        user_csv_path = f"/content/{file_name}"
        
        # Preview data and get column info
        try:
            df = pd.read_csv(user_csv_path, nrows=5)
            print(f"\nUploaded: {file_name}")
            print("\nFirst 5 rows:")
            print(df)
            print(f"\nColumns: {list(df.columns)}")
            
            # Ask for column names
            target_column = input("\nEnter target column name: ").strip()
            if not target_column:
                print("No target column specified. Using first numeric column.")
                numeric_cols = df.select_dtypes(include=['number']).columns
                if len(numeric_cols) > 0:
                    target_column = numeric_cols[0]
                    print(f"Using '{target_column}' as target column.")
                else:
                    print("No numeric columns found. Using first column.")
                    target_column = df.columns[0]
                    print(f"Using '{target_column}' as target column.")
            
            time_column = input("Enter time column name (or press Enter if none): ").strip() or None
            if time_column and time_column not in df.columns:
                print(f"Warning: Column '{time_column}' not found in CSV. Setting to None.")
                time_column = None
            
            print(f"\nUsing:\n- Data: {file_name}\n- Target column: {target_column}\n- Time column: {time_column if time_column else 'None'}")
        except Exception as e:
            print(f"Error reading CSV: {e}")
            print("Falling back to synthetic data.")
            choice = "3"
    else:
        print("No file uploaded. Falling back to synthetic data.")
        choice = "3"
elif choice == "2":
    print("Will attempt to download NASA POWER Austin dataset...")
    # Use the existing download functionality from colab_utils
    from colab_utils import download_sample_dataset
    try:
        user_csv_path = download_sample_dataset("nasa_power_austin_hourly")
        if user_csv_path:
            target_column = "T2M"
            time_column = "time"
            print(f"Downloaded dataset: {user_csv_path}")
            print(f"Target column: {target_column}")
            print(f"Time column: {time_column}")
        else:
            print("Download failed. Falling back to synthetic data.")
            choice = "3"
    except Exception as e:
        print(f"Download error: {e}")
        print("Falling back to synthetic data.")
        choice = "3"
else:
    print("Using synthetic demo data...")
    # Synthetic data will be created in the configuration step

print("\n" + "="*60)

## 3. Configuration

We'll configure a fast demo run with reduced parameters suitable for Colab's runtime limits.

In [ ]:
import yaml
import json
from pathlib import Path

# Determine final data path and columns
if choice == "1" and user_csv_path:
    # User uploaded CSV
    final_data_path = user_csv_path
    final_target_col = target_column
    final_time_col = time_column
elif choice == "2" and user_csv_path:
    # Downloaded NASA dataset
    final_data_path = user_csv_path
    final_target_col = "T2M"
    final_time_col = "time"
else:
    # Synthetic data - create it now
    from colab_utils import create_synthetic_dataset
    final_data_path = create_synthetic_dataset()
    final_target_col = "value"
    final_time_col = "time"
    print(f"Created synthetic dataset: {final_data_path}")

# Demo configuration
demo_config = {
    'data_path': final_data_path,
    'target_col': final_target_col,
    'time_col': final_time_col if final_time_col else None,  # Empty string if None
    'L': 30,  # Lookback window
    'H': 5,   # Forecast horizon
    'epochs': 15,  # Reduced for demo
    'batch_size': 64,
    'max_train_windows': 1000,  # Limit for speed
    'noise_levels': '0,0.05,0.1',
    'skip_classical': True,  # Skip slow ARIMA/ARMA for demo
    'skip_noise_robustness': False,
    'gan_variant': 'wgan-gp',
    'units_g': 64,
    'units_d': 64,
    'g_layers': 1,
    'd_layers': 1,
    'dropout': 0.1,
    'lr_g': 5e-4,
    'lr_d': 5e-4,
    'lambda_reg': 0.5,
    'results_dir': './results/colab_demo',
    'deterministic': False,
    'seed': 42,
}

# Save config for reference
config_path = Path('demo_config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(demo_config, f)

print("Demo configuration:")
for key, value in demo_config.items():
    print(f"  {key}: {value}")

# Note about time column
if not demo_config['time_col']:
    print("\nNote: No time column specified. The model will use integer indexing.")

## 4. Run RGAN Training

Now we'll run the main training pipeline with our chosen data and configuration.

In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Build command from config
cmd = [
    sys.executable, "-u", "-m", "rgan.scripts.run_training",
    "--csv", demo_config['data_path'],
    "--target", demo_config['target_col'],
]

# Add time column if specified
if demo_config['time_col'] and demo_config['time_col'].strip():
    cmd.extend(["--time_col", demo_config['time_col']])

# Add remaining parameters
cmd.extend([
    "--L", str(demo_config['L']),
    "--H", str(demo_config['H']),
    "--epochs", str(demo_config['epochs']),
    "--batch_size", str(demo_config['batch_size']),
    "--max_train_windows", str(demo_config['max_train_windows']),
    "--noise_levels", demo_config['noise_levels'],
    "--gan_variant", demo_config['gan_variant'],
    "--units_g", str(demo_config['units_g']),
    "--units_d", str(demo_config['units_d']),
    "--g_layers", str(demo_config['g_layers']),
    "--d_layers", str(demo_config['d_layers']),
    "--dropout", str(demo_config['dropout']),
    "--lr_g", str(demo_config['lr_g']),
    "--lr_d", str(demo_config['lr_d']),
    "--lambda_reg", str(demo_config['lambda_reg']),
    "--results_dir", demo_config['results_dir'],
    "--seed", str(demo_config['seed']),
])

if demo_config['skip_classical']:
    cmd.append("--skip_classical")

if demo_config['skip_noise_robustness']:
    cmd.append("--skip_noise_robustness")

if demo_config['deterministic']:
    cmd.append("--deterministic")

print("Running command:")
print(" ".join(cmd))
print("\n" + "="*80)

# Execute training
result = subprocess.run(cmd, capture_output=True, text=True)

# Print output
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

print(f"\nTraining completed with exit code: {result.returncode}")
if result.returncode != 0:
    print("Training failed. Check error messages above.")
else:
    print("Training successful!")

## 5. Load and Display Results

Let's load the generated metrics and visualize the outcomes.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'notebook'

results_dir = Path(demo_config['results_dir'])

# Load metrics
metrics_path = results_dir / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print("Key Metrics Summary:")
    print("=" * 50)
    
    # Extract RGAN metrics
    if 'rgan' in metrics:
        rgan_metrics = metrics['rgan']
        print(f"RGAN Test RMSE: {rgan_metrics.get('test_stats', {}).get('rmse', 'N/A'):.6f}")
        print(f"RGAN Test MAE: {rgan_metrics.get('test_stats', {}).get('mae', 'N/A'):.6f}")
    
    # Extract other model metrics
    model_keys = [k for k in metrics.keys() if k not in ('L', 'H', 'dataset', 'environment')]
    for model in model_keys:
        if model != 'rgan' and isinstance(metrics[model], dict):
            stats = metrics[model].get('test_stats', {})
            if stats:
                print(f"{model.upper()} Test RMSE: {stats.get('rmse', 'N/A'):.6f}")
else:
    print(f"Metrics file not found at {metrics_path}")

In [ ]:
# Load noise robustness table if available
noise_table_path = results_dir / 'noise_robustness_table.csv'
if noise_table_path.exists():
    df_noise = pd.read_csv(noise_table_path)
    print("\nNoise Robustness Table:")
    print(df_noise.to_string(index=False))
    
    # Plot noise robustness
    if 'noise_level' in df_noise.columns and 'rmse' in df_noise.columns:
        plt.figure(figsize=(10, 6))
        for model in df_noise['model'].unique():
            model_data = df_noise[df_noise['model'] == model]
            plt.plot(model_data['noise_level'], model_data['rmse'], marker='o', label=model)
        plt.xlabel('Noise Level (σ)')
        plt.ylabel('RMSE')
        plt.title('Model Robustness to Additive Gaussian Noise')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print("Noise robustness table not generated (may have been skipped)")

In [ ]:
# Load and display training curves
training_plots = list(results_dir.glob('*training_curves*.png'))
if training_plots:
    print(f"\nTraining plots generated: {[p.name for p in training_plots]}")
    for plot_path in training_plots[:2]:  # Show first two
        from IPython.display import Image
        display(Image(filename=str(plot_path)))
else:
    print("No training plots found")

In [ ]:
# Interactive controls for real-time parameter adjustment (optional)
import ipywidgets as widgets
from IPython.display import display, HTML

print("\n=== Interactive Controls ===")
print("Adjust parameters and re-run training cells if needed.")

# Create sliders for key parameters
epoch_slider = widgets.IntSlider(
    value=15,
    min=5,
    max=50,
    step=5,
    description='Epochs:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

batch_slider = widgets.IntSlider(
    value=64,
    min=16,
    max=256,
    step=16,
    description='Batch size:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

noise_slider = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='Max noise:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f'
)

# Display the sliders
display(HTML("<h4>Adjust Training Parameters</h4>"))
display(widgets.VBox([epoch_slider, batch_slider, noise_slider]))

# Update demo_config if user wants to apply changes
print("\nTo apply these changes, modify demo_config in the cell above:")
print(f"  demo_config['epochs'] = {epoch_slider.value}")
print(f"  demo_config['batch_size'] = {batch_slider.value}")
print(f"  demo_config['noise_levels'] = f'0,{{noise_slider.value/2:.2f}},{{noise_slider.value:.2f}}'")

## 6. Optional: Augmentation Experiment

If you have time and want to see synthetic data generation, run this optional augmentation experiment.

In [ ]:
# Only run if training was successful and we have a model
augmentation_cmd = [
    sys.executable, "-u", "-m", "rgan.scripts.run_augmentation",
    "--csv", demo_config['data_path'],
    "--target", demo_config['target_col'],
]

# Add time column if specified
if demo_config['time_col']:
    augmentation_cmd.extend(["--time_col", demo_config['time_col']])

augmentation_cmd.extend([
    "--L", str(demo_config['L']),
    "--H", str(demo_config['H']),
    "--results_from", str(results_dir),
    "--results_dir", str(results_dir / "augmentation"),
    "--skip_timegan",  # Skip TimeGAN for speed
    "--nn_epochs", "10",  # Reduced for demo
    "--nn_patience", "5",
])

print("Running augmentation experiment...")
print(" ".join(augmentation_cmd))

aug_result = subprocess.run(augmentation_cmd, capture_output=True, text=True)
print(aug_result.stdout)
if aug_result.stderr:
    print("STDERR:", aug_result.stderr)

In [ ]:
# Display augmentation results if available
aug_dir = results_dir / "augmentation"
if aug_dir.exists():
    # Show synthetic quality table
    quality_table = aug_dir / "synthetic_quality_table.csv"
    if quality_table.exists():
        df_quality = pd.read_csv(quality_table)
        print("\nSynthetic Quality Metrics:")
        print(df_quality.to_string(index=False))
    
    # Show augmentation comparison table
    aug_table = aug_dir / "data_augmentation_table.csv"
    if aug_table.exists():
        df_aug = pd.read_csv(aug_table)
        print("\nData Augmentation Results (Real vs Mixed):")
        print(df_aug.to_string(index=False))

## 7. Save Results to Google Drive (Optional)

If you mounted Google Drive earlier, you can save your results there for later access.

In [ ]:
if IN_COLAB and setup_result.get('drive_path'):
    from google.colab import drive
    import shutil
    
    drive_results = setup_result['drive_path'] / 'RGAN_Results'
    drive_results.mkdir(exist_ok=True)
    
    # Copy results directory
    dest = drive_results / results_dir.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(results_dir, dest)
    
    print(f"Results saved to Google Drive: {dest}")
else:
    print("Google Drive not mounted or not in Colab. Results remain in Colab's temporary storage.")

## 8. Next Steps & Documentation

### Running Full Experiments
For more serious experiments:
1. Use full datasets (download via `scripts/fetch_natural_datasets.py`)
2. Increase training epochs (e.g., `--epochs 200`)
3. Enable all baselines (remove `--skip_classical`)
4. Use larger model sizes (`--units_g 128 --units_d 128 --g_layers 2 --d_layers 2`)

### Available Datasets
The project includes scripts to download multiple real-world noisy datasets:
- NASA POWER meteorological data (Austin, Phoenix, Denver)
- NOAA tide and buoy data
- USGS streamflow data
- Beijing air quality, household power, gas sensors, etc.

### Project Structure
- `src/rgan/` - Core implementation
- `scripts/` - Data fetching and utilities
- `cloud/` - AWS SageMaker deployment
- `docs/` - Detailed documentation

### CLI Usage Examples
```bash
# Full training run
rgan-train --csv data/nasa_power_austin_hourly/nasa_power_austin_hourly.csv --target T2M --epochs 200

# Noise robustness sweep
rgan-train --csv data/beijing_air/beijing_air.csv --target PM2.5 --noise_levels 0,0.01,0.05,0.1,0.2

# Augmentation experiment
rgan-augment --csv data/household_power/household_power.csv --results_from results/experiment_20250101_120000
```

### Troubleshooting
- **CUDA out of memory**: Reduce `--batch_size` or `--max_train_windows`
- **Training too slow**: Use `--skip_classical` and `--skip_noise_robustness`
- **Symlink errors**: Ignore (non-critical convenience feature)
- **CSV upload issues**: Ensure your CSV has a header row and numeric target column

---